This file contains the mathematical justifications for tensor operators implemented in code, and their gradient update policy.

**A preface on bilinear operators.**

Recall that a bilinear operator is a operation that is linear with respects to both tensor inputs. Meaning:
$\bold{A} * (c\bold{C} +c\bold{D}) = b(\bold{A} * \bold{B}) +c(\bold{A} * \bold{C})$

$(a\bold{A} + b\bold{B}) * C = a(\bold{A} * \bold{C}) + b(\bold{B} * \bold{C})$

Although the rules seems simple, they are the backbone of the most essential operations in ML: Matrix multiplication, convlution, flattening, etc. There are many layers that use bilinear operators.

It is worthwhile to consider what bilinear operators are in ML.

Here, bilinear operators always taking in two tensors and output a tensor. 

There are two essential facts about bilinear operators on tensors we need to grasp:
1. The space of all possible linear transforms is itself a vector space.

*proof*

Consider tensor spaces $V_A, V_B$ and the set of all possible linear operators that binds them: $\Omega = \{T: T =  V_A \rightarrow V_B\}$

$\Omega is closed under scalar multiplication, as the operation $aT[\bold{v}]$ for $\bold{v} \in V_A$ is itself a linear operator and therefore part of $\Omega$

Similarly $T_1[\bold{v}] + T_2[\bold{v}]$ is a linear operator and therefore must be in $\Omega$

This sets the background for the next major fact


2. All possible linear transforms from one tensor space to another can be written as the bilinear operator of two tensors.

This one is the absolute backbone of matmul. Consider the basis vectors in $V_A$ being $\bold{e_n}, n > 0$

Then $v =\sum_j a_j\bold{e_j}$ and $T[\bold{v}] = \sum a_jT[\bold e_j]$

Now denote a tensor $\bold{G}$ such that $\bold{G}_i = a_jT[\bold e_j]$. Remember, tensors are multdimensional arrays. We are basically saying $\bold{G}$ is a multidimensional array with elements equal to $a_jT[\bold e_j]$

Another tensor $\bold{a}$ such that $\bold{a}_j = a_j$. This might look weird, but its just the vector representation of all the scalar constants.

Then define a linear operator $\bold{a} * \bold{G} = \sum_j a_j\bold{e_j}$. 

While this might seem trivial, abstracting truths allows us to scale up intuitive ideas to create unbelievably large models. 

For example, the bilinear operator of matrix multiplication fully encapsulates linear transforms from $R^n$ to $R^m$. 

Similarly, the convolution operation from CNN's is can also be represented as a bilinear operator. 

In pratical terms, bilinear operators are the actual machinery that compute the otherwise abstract linear transforms. 

The standard operator is a procedure that generalizes matrix multiplication to tensors. Its explicit details are not necessary to understand right now, just know that its the standard bilinear operator used to define linear operators and encapsulate their dual behavior. 

In the next section, we will consider how to perform backdrop on the most generic of bilinear operators: matrix mult.

**Key formula** Consider: $\bold{A} \cdot (\bold{B}\bold{C})$, where $\bold{A}$,$\bold{B}$,$\bold{C}$ are matrices. We want to extract $\bold{B}$ and $\bold{C}$ out of the expression.

We can use summation notation to explicitly write this out:

$\bold{A} \cdot (\bold{B}\bold{C}) = \sum_i\sum_k a_{ik} \sum_j b_{ij}c_{jk}$


$ = \sum_i\sum_k\sum_j a_{ik}b_{ij}c_{jk}$

Now we do some big brain thinking: what is $\bold{C}^T$? Each i,j element in the transpose is flipped across the major diagonal meaning:

$\bold{C}^T_{kj} = c_{jk}$

$ = \sum_i\sum_k\sum_j a_{ik}\bold{C}^T_{kj}b_{ij}$

$ = \sum_i\sum_j b_{ij}\sum_k (a_{ik}\bold{C}^T_{kj})$
$ = \bold{B} \cdot {\bold{A}\bold{C}^T}$

Going back to  $\sum_i\sum_k\sum_j a_{ik}b_{ij}c_{jk}$ we can do a similar thing to $\bold{B}$: $\bold{B}^T_{ji} = b_{ij}$


$\sum_i\sum_k\sum_j a_{ik}\bold{B}^T_{ji}c_{jk} = \sum_j\sum_k c_{jk}\sum_i (\bold{B}^T_{ji}a_{ik})$

$ = \bold{C} \cdot (\bold{B}^T \bold{A})$

In summary we have 
$$\bold{A} \cdot (\bold{B}\bold{C}) = (\bold{A}\bold{C}^T) \cdot \bold{B} = (\bold{B}^T \bold{A}) \cdot \bold{C}$$

The easiest way to remember this is to look at the relative placements: $\bold{C}$ is the on right side of the matmul so it goes to the right of $\bold{A}$, and $\bold{B}$ is on the left side of matmul so its transpose goes to the left of $\bold{A}$


**Matrix Multiplication**

Consider the layer $\bold{T_{data}} = \bold{T_{input,1}}\bold{T_{input,2}}$. 

By the formula: $\frac{\partial \bold{E}}{\partial \bold{T_{input,2}}} \cdot d\bold{T_{input,2}}= \frac{\partial \bold{E}}{\partial \bold{T_{data}}} \cdot (d\bold{T_{data}}) 
= \frac{\partial \bold{E}}{\partial \bold{T_{data}}} \cdot (\bold{T_{input,1}}d\bold{T_{input,2}})$

By the previous formula:

$\frac{\partial \bold{E}}{\partial \bold{T_{input,2}}} \cdot (d\bold{T_{input,2}}) = \bold{T_{input,1}}^T \frac{\partial \bold{E}}{\partial \bold{T_{data}}}T \cdot d\bold{T_{input,2}}$

$\frac{\partial \bold{E}}{\partial \bold{T_{input,2}}} = \bold{T_{input,1}}^T \frac{\partial \bold{E}}{\partial \bold{T_{data}}}$

Similarly, for $\frac{\partial \bold{E}}{\partial \bold{T_{input,1}}}$ we get:
$\frac{\partial \bold{E}}{\partial \bold{T_{input,1}}} \cdot d\bold{T_{input,1}} =  \frac{\partial \bold{E}}{\partial \bold{T_{data}}} \cdot (d\bold{T_{input,1}}\bold{T_{input,2}})$

$\implies \frac{\partial \bold{E}}{\partial \bold{T_{input,1}}} = \frac{\partial \bold{E}}{\partial \bold{T_{input,1}}}\bold{T_{input,2}}^T$



**Addition**
Consider the layer $\bold{T_{data}} = \bold{T_{input,1}} + \bold{T_{input,2}}$

Then:
$\frac{\partial E}{\partial T_{input,1}} \cdot dT_{input,1}= \frac{\partial E}{\partial T_{data}} \cdot (d\bold{T_{input,1}} + 0)$

$\implies \frac{\partial E}{\partial T_{input,1}} = \frac{\partial E}{\partial T_{data}}$

Similarly since addition is commutative: $\frac{\partial E}{\partial T_{input,2}} = \frac{\partial E}{\partial T_{data}}$

**Scalar Multiplication:**
$\bold{T_{data}} = c\bold{T_{input}}$

$\frac{\partial E}{\partial T_{input,1}} \cdot dT_{input,1}= c\frac{\partial E}{\partial T_{data}} \cdot (d\bold{T_{input,1}})$

$\frac{\partial E}{\partial T_{input,1}} = c\frac{\partial E}{\partial T_{data}}$

*what if you want the scalar to be a parameter*?

Note that c is a scalar so its dot product is just regular mult

$\frac{\partial E}{\partial c}dc= \frac{\partial E}{\partial T_{data}} \cdot (dc\bold{T_{input}}) = (\frac{\partial E}{\partial T_{data}}) \cdot (\bold{T_{input}})dc$

$\frac{\partial E}{\partial c} = (\frac{\partial E}{\partial T_{data}}) \cdot (\bold{T_{input}})$



**Activation Functions**
They take the form $\bold{T} = f(\bold{T_1})$, where $f$ is applied to each component

I'm using a different notation for the gradient shadow (gradient notation) because its easier to type and more literal. ill change the previous partials in another update.

$\nabla_{T_1} E \cdot d\bold{T_1}= \nabla_{T} E \cdot df = \nabla_{T} E \cdot (f'(\bold{T_1}) * d{\bold{T_1}})$, where $*$ denotes element wise multiplication.

$\nabla_{T} E \cdot (f'(\bold{T_1}) * d{\bold{T_1}}) = \nabla_{T} E * f'(\bold{T_1}) \cdot d{\bold{T_1}}$

The fact above can be easily derived by looking at the element wise addition thru summation notation.

Therefore:

$$\nabla_{T_1} E = \nabla_{T} E * f'(\bold{T_1}) $$

Where $*$ denotes element wise multiplication